In [40]:
from z3 import *
from itertools import combinations
import time
from constraints import *
from utils import *

In [41]:
exactly_one = exactly_one_he
at_most_one = at_most_one_seq
at_most_k = at_most_k_seq
at_least_k = at_least_k_seq

In [42]:
def STS_SAT(n, time_limit=300):
    start_time = time.time()
    
    W = n - 1
    P = n // 2
    M = n * (n - 1) // 2
    T1, T2 = build_inverse_tables(n)

    print(f"Solving for {n} teams ===")
    print(f"Teams: {n}, Weeks: {W}, Periods: {P}, Matches: {M}")
    print(f"Time limit: {time_limit}s")

    # PHASE 1: Find a feasible schedule
    print("\n=== PHASE 1: Finding feasible schedule ===")

    # Generate fixed schedule from circle method
    circle_schedule_weeks = {}
    circle_schedule_full = circle_method_fixed_schedule(n)
    for (w, p), m in circle_schedule_full.items():
        if w not in circle_schedule_weeks:
            circle_schedule_weeks[w] = []
        circle_schedule_weeks[w].append(m)
    
    s = Solver()

    # Variable
    # match_period[m][p] is True if match m is in period p
    match_period = [[Bool(f"match_{m}_period_{p}") 
                     for p in range(P)] 
                     for m in range(M + 1)]
    
    # Constraints
    # 1. Each match from the circle schedule must be assigned to exactly one period
    for w in range(W):
        matches_in_week = circle_schedule_weeks[w]
        for m in matches_in_week:
            s.add(exactly_one(match_period[m], f"one_period_m{m}"))

    # 2. Each period must contain exactly one match from each week
    for w in range(W):
        for p in range(P):
            matches_in_week = circle_schedule_weeks[w]
            s.add(exactly_one([match_period[m][p] for m in matches_in_week], f"one_match_per_slot_w{w}_p{p}"))
    
    # 3. Each team plays at most two match in the same period
    for p in range(P):
        for t in range(1, n + 1):
            team_plays_in_period = []
            for w in range(W):
                matches_in_week = circle_schedule_weeks[w]
                for m in matches_in_week:
                    if T1[m] == t or T2[m] == t:
                        team_plays_in_period.append(match_period[m][p])
            s.add(at_most_k(team_plays_in_period, 2, f"at_most_two_play_t{t}_p{p}"))

    # 4. Each team has exactly one defective period (appears exactly once in that period)
    for t in range(1, n + 1):
        defective_indicators = []
        for p in range(P):
            defective_p = Bool(f"defective_p{p}_t{t}")
            defective_indicators.append(defective_p)
            
            appearances = []
            for w in range(W):
                matches_in_week = circle_schedule_weeks[w]
                for m in matches_in_week:
                    if T1[m] == t or T2[m] == t:
                        appearances.append(match_period[m][p])
            
            s.add(defective_p == exactly_one(appearances, f"defective_p{p}_t{t}_implies_exactly_one"))
        
        s.add(exactly_one(defective_indicators, f"exactly_one_defective_period_t{t}"))

    # Symmetry breaking constraints
    # s.add(match_period[circle_schedule_full[(0,0)]][0])
    

    print("Solving scheduling phase...")
    schedule_result = s.check()
    phase1_time = time.time() - start_time
    
    if schedule_result != sat:
        print(f"No feasible schedule found in {phase1_time:.2f}s")
        return None
    
    schedule_model = s.model()
    print(f"Feasible schedule found in {phase1_time:.2f}s")
    
    # Extract the fixed schedule
    fixed_schedule = {}
    for w in range(W):
        matches_in_week = circle_schedule_weeks[w]
        for m in matches_in_week:
            for p in range(P):
                if is_true(schedule_model.evaluate(match_period[m][p])):
                    fixed_schedule[(w, p)] = m
                    break

    # PHASE 2: Optimize home/away assignments using binary search
    print("\n=== PHASE 2: Optimizing home/away assignments ===")
    
    remaining_time = time_limit - phase1_time
    if remaining_time <= 0:
        print("No time remaining for optimization")
        swap = [BoolVal(False) for _ in range(M + 1)]
        return schedule_model, fixed_schedule, None, swap

    # Compute the initial imbalance
    initial_imbalance, _ = calculate_imbalance(n, fixed_schedule, None, None)
    print(f"Initial imbalance (no swaps): {initial_imbalance}")

    # Binary search bounds
    lower_bound = 0
    upper_bound = initial_imbalance
    
    best_solution = None
    best_imbalance = initial_imbalance

    # Swap variables
    swap = [Bool(f"swap_m{m}") for m in range(M + 1)]

    # Pre-compute the home_vars for each team
    team_home_vars = [[] for t in range(n + 1)]
    for t in range(1, n + 1):
        vars_t = []
        for (w, p), m in fixed_schedule.items():
            if T1[m] == t:
                vars_t.append(Not(swap[m]))
            elif T2[m] == t:
                vars_t.append(swap[m])
        team_home_vars[t] = vars_t


    max_team_imbalance = min(n - 1, initial_imbalance)

    print(f"Starting binary search optimization (bounds: {lower_bound}-{upper_bound})")
    
    while lower_bound <= upper_bound and time.time() - start_time < time_limit - 1:
        mid = (lower_bound + upper_bound) // 2
        print(f"Trying total imbalance <= {mid}")
        
        opt_solver = Solver()
        
        # List of bits for the total imbalance (is the sum of all team imbalances)
        total_imbalance_bits = []
        
        for t in range(1, n + 1):
            # Count home games for team t
            home_count = sum([If(var, 1, 0) for var in team_home_vars[t]])
            
            # Create the imbalance bits directly from home_count
            imbalance_bits = [Bool(f"team{t}_imb_bit_{i}") for i in range(max_team_imbalance)]
            
            # Model the imbalance directly from the home_count value
            # imbalance >= k iff abs(2*home_count - (n-1)) >= k+1
            raw_diff = 2 * home_count - (n-1)
            for k in range(1, max_team_imbalance + 1):
                pos_cond = And(raw_diff >= 0, raw_diff >= k+1)
                neg_cond = And(raw_diff < 0, raw_diff <= -(k+1))
                opt_solver.add(imbalance_bits[k-1] == Or(pos_cond, neg_cond))

            # 4. Add ordering constraint for imbalance_bits
            for i in range(1, len(imbalance_bits)):
                opt_solver.add(Or(Not(imbalance_bits[i]), imbalance_bits[i-1]))
            
            # Add these bits to total imbalance calculation
            total_imbalance_bits.extend(imbalance_bits)
        
        # Now we constrain that the sum of all team imbalances <= mid
        total_count = sum([If(bit, 1, 0) for bit in total_imbalance_bits])
        opt_solver.add(total_count <= mid)

        # Symmetry breaking
        # opt_solver.add(Not(swap[1]))
        
        # Solve
        opt_result = opt_solver.check()
        
        if opt_result == sat:
            swap_model = opt_solver.model()
            actual_imbalance, team_imbalances = calculate_imbalance(n, fixed_schedule, swap_model, swap)
            
            print(f"Solution found with actual imbalance {actual_imbalance} (target was <= {mid})")
            
            if actual_imbalance <= best_imbalance:
                best_solution = (schedule_model, fixed_schedule, swap_model, swap)
                best_imbalance = actual_imbalance
            
            if actual_imbalance == 0:
                print("Optimal solution found!")
                break
                
            # Update bounds
            upper_bound = min(mid - 1, actual_imbalance - 1)
        else:
            print(f"No solution with imbalance <= {mid}")
            lower_bound = mid + 1
    
    total_time = time.time() - start_time
    print(f"\n=== FINAL RESULTS ===")
    print(f"Total time: {total_time:.2f}s")
    print(f"Phase 1 (scheduling): {phase1_time:.2f}s")
    print(f"Phase 2 (optimization): {total_time - phase1_time:.2f}s")
    
    if best_solution:
        print(f"Best imbalance found: {best_imbalance}")
        return best_solution
    else:
        print("Using basic solution with no swaps")
        swap = [BoolVal(False) for _ in range(M + 1)]
        return schedule_model, fixed_schedule, None, swap


In [43]:
n = 18
print(f"Solving tournament scheduling for {n} teams...")
print(f"Teams: {n}, Weeks: {n-1}, Periods: {n//2}, Matches: {n*(n-1)//2}")

start_time = time.time()
result = STS_SAT(n, time_limit=300)
elapsed = time.time()-start_time

if result:
    schedule_model, fixed_schedule, swap_model, swap = result
    
    print(f"\n--- SCHEDULE for n={n} ---")
    print_schedule_optimized(n, fixed_schedule, swap_model, swap)
    
    # Verify the solution
    actual_imbalance, team_imbalances = calculate_imbalance(n, fixed_schedule, swap_model, swap)
    print(f"\nVerification - Total imbalance: {actual_imbalance}")
    for t, imb in enumerate(team_imbalances, 1):
        print(f"Team {t}: imbalance = {imb}")
else:
    print(f"Failed to find solution for n={n}")

print(f"Total time: {elapsed:.2f}s")

Solving tournament scheduling for 18 teams...
Teams: 18, Weeks: 17, Periods: 9, Matches: 153
Solving for 18 teams ===
Teams: 18, Weeks: 17, Periods: 9, Matches: 153
Time limit: 300s

=== PHASE 1: Finding feasible schedule ===
Solving scheduling phase...
Feasible schedule found in 31.05s

=== PHASE 2: Optimizing home/away assignments ===
Initial imbalance (no swaps): 144
Starting binary search optimization (bounds: 0-144)
Trying total imbalance <= 72
Solution found with actual imbalance 0 (target was <= 72)
Optimal solution found!

=== FINAL RESULTS ===
Total time: 31.90s
Phase 1 (scheduling): 31.05s
Phase 2 (optimization): 0.84s
Best imbalance found: 0

--- SCHEDULE for n=18 ---
          Week 1     Week 2     Week 3     Week 4     Week 5     Week 6     Week 7     Week 8     Week 9     Week 10    Week 11    Week 12    Week 13    Week 14    Week 15    Week 16    Week 17    
Period 1  7 vs 12    8 vs 13    1 vs 5     9 vs 16    1 vs 9     18 vs 6    14 vs 17   6 vs 10    7 vs 11    15 vs